# 05 — Two-Tower Retrieval — MerRec

Bản Kaggle **FULL / RAM-safe / resumable**, thiết kế cho bộ `MerRec Retrieval Training`.

- TRAIN học model.
- VAL chọn best checkpoint.
- TEST chỉ đánh giá cuối.
- User tower: sparse user embedding.
- Item tower: hashed metadata embeddings + MLP (không dùng item-ID embedding 29M item).
- In-batch negatives + symmetric InfoNCE.
- Đọc TRAIN theo Parquet row-group, không load 100M+ pairs vào RAM.
- Checkpoint/resume.
- Build FAISS IVF-PQ toàn bộ TRAIN-item universe.
- Cuối notebook tạo `/kaggle/working/TwoTower_serving_package.zip`.

**Nên bật GPU T4/P100.**

In [1]:
# Cell 1 — Imports / dependencies
import os, sys, gc, json, math, time, random, shutil, zipfile, subprocess
from pathlib import Path
from datetime import datetime

def ensure(pkg, pip_name=None):
    try: return __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])
        return __import__(pkg)

np = ensure("numpy")
pl = ensure("polars")
pa = ensure("pyarrow")
psutil = ensure("psutil")
torch = ensure("torch")
try:
    import faiss
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    import faiss

import pyarrow.parquet as pq
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD, AdamW

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.1 MB/s eta 0:00:00
Torch: 2.10.0+cu128 | CUDA: True
Tesla T4


In [2]:
# Cell 2 — Config
SEED = 20260918
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = {
    "embed_dim": 64, "feature_dim": 16, "hidden_dim": 256,
    "dropout": 0.10, "temperature": 0.07,
    "epochs": 15,
    "batch_size": 2048 if torch.cuda.is_available() else 512,
    "user_lr": 0.08, "dense_lr": 3e-4, "weight_decay": 1e-5,
    "grad_clip": 1.0, "checkpoint_minutes": 35, "patience": 2,
    "val_users": 3000, "test_users": 5000,
    "val_random_items": 500000,
    "eval_ks": [10,20,100], "retrieve_extra": 300,
    "hash_buckets": {
        "product_id": 2**20, "brand": 2**18,
        "category0": 2**16, "category1": 2**16, "category2": 2**16,
        "condition": 2**10, "shipper": 2**16
    },
    "faiss_nlist": 4096, "faiss_m": 16, "faiss_nbits": 8,
    "faiss_nprobe": 24, "faiss_train_sample": 250000,
    "faiss_batch": 65536
}
assert CFG["embed_dim"] % CFG["faiss_m"] == 0
print("DEVICE:", DEVICE)
print(json.dumps(CFG, indent=2))

DEVICE: cuda
{
  "embed_dim": 64,
  "feature_dim": 16,
  "hidden_dim": 256,
  "dropout": 0.1,
  "temperature": 0.07,
  "epochs": 15,
  "batch_size": 2048,
  "user_lr": 0.08,
  "dense_lr": 0.0003,
  "weight_decay": 1e-05,
  "grad_clip": 1.0,
  "checkpoint_minutes": 35,
  "patience": 2,
  "val_users": 3000,
  "test_users": 5000,
  "val_random_items": 500000,
  "eval_ks": [
    10,
    20,
    100
  ],
  "retrieve_extra": 300,
  "hash_buckets": {
    "product_id": 1048576,
    "brand": 262144,
    "category0": 65536,
    "category1": 65536,
    "category2": 65536,
    "condition": 1024,
    "shipper": 65536
  },
  "faiss_nlist": 4096,
  "faiss_m": 16,
  "faiss_nbits": 8,
  "faiss_nprobe": 24,
  "faiss_train_sample": 250000,
  "faiss_batch": 65536
}


In [3]:
# Cell 3 — Resolve paths + helpers
KINPUT = Path("/kaggle/input"); WORK = Path("/kaggle/working")
ROOT = WORK/"merrec_models"/"TwoTower"
for d in ["processed","features","checkpoints","model","index","metrics","status","serving"]:
    (ROOT/d).mkdir(parents=True, exist_ok=True)

def find_one(name):
    ms = list(KINPUT.rglob(name))
    ms2 = [p for p in ms if "merrec" in str(p).lower()]
    ms = ms2 or ms
    if not ms: raise FileNotFoundError(name)
    return ms[0]

CF = find_one("cf_train.parquet")
MD = CF.parent
USER_MAP = MD/"train_user_map.parquet"
ITEM_MAP = MD/"train_item_map.parquet"
CAT = (list(KINPUT.rglob("item_features.parquet")) or [MD/"item_catalog_train.parquet"])[0]
INTDIR = MD/"interactions"
VAL_GLOB = str(INTDIR/"split=val"/"*.parquet")
TEST_GLOB = str(INTDIR/"split=test"/"*.parquet")

def schema(p): return pq.read_schema(p).names
def col(names, opts, req=True):
    m={x.lower():x for x in names}
    for x in opts:
        if x.lower() in m: return m[x.lower()]
    if req: raise KeyError((opts,names))
    return None
def atomic_json(o,p):
    p=Path(p); t=Path(str(p)+".tmp"); t.write_text(json.dumps(o,indent=2,default=str),encoding="utf-8"); os.replace(t,p)
def atomic_torch(o,p):
    p=Path(p); t=Path(str(p)+".tmp"); torch.save(o,t); os.replace(t,p)
def res():
    vm=psutil.virtual_memory(); du=shutil.disk_usage(WORK)
    print(f"RAM available={vm.available/2**30:.2f}GB | working free={du.free/2**30:.2f}GB")

# Recover previous checkpoint if a saved Kaggle output is attached as Input.
for name in ["twotower_latest.pt","twotower_best.pt"]:
    dst=ROOT/"checkpoints"/name
    if not dst.exists():
        ms=[p for p in KINPUT.rglob(name) if "tower" in str(p).lower()]
        if ms:
            shutil.copy2(ms[0],dst); print("Recovered",ms[0])

print("CF:",CF); print("CAT:",CAT); print("ROOT:",ROOT); res()

CF: /kaggle/input/deleted-dataset/merrec_retrieval_training/model_data/cf_train.parquet
CAT: /kaggle/input/deleted-dataset/merrec_retrieval_training/model_specific/item_features.parquet
ROOT: /kaggle/working/merrec_models/TwoTower
RAM available=29.63GB | working free=19.50GB


In [4]:
# Cell 4 — Schemas / dimensions
un=schema(USER_MAP); inn=schema(ITEM_MAP); cn=schema(CF)
UID=col(un,["user_id","userid","user"]); UIX=col(un,["user_idx","user_index","uid","idx"])
IID=col(inn,["item_id","itemid","item"]); IIX=col(inn,["item_idx","item_index","iid","idx"])
CFU=col(cn,["user_id",UID]); CFI=col(cn,["item_id",IID])
SCORE=col(cn,["implicit_score","score","weight","interactions"])

N_USERS=int(pl.scan_parquet(USER_MAP).select(pl.len()).collect().item())
N_ITEMS=int(pl.scan_parquet(ITEM_MAP).select(pl.len()).collect().item())
N_PAIRS=int(pl.scan_parquet(CF).select(pl.len()).collect().item())
print(f"users={N_USERS:,} items={N_ITEMS:,} pairs={N_PAIRS:,}")

users=2,581,378 items=27,328,461 pairs=109,718,519


In [5]:
# Cell 5 — Build RAM-safe integer TRAIN pairs
TRAIN = ROOT/"processed"/"train_pairs.parquet"
OK = ROOT/"status"/"_TRAIN_PAIRS_SUCCESS.json"

def valid_parquet(p,n=1):
    try: return Path(p).exists() and pq.ParquetFile(p).metadata.num_rows>=n
    except: return False

if not (valid_parquet(TRAIN,1000) and OK.exists()):
    u=pl.scan_parquet(USER_MAP).select([pl.col(UID).alias("_u"),pl.col(UIX).cast(pl.UInt32).alias("user_idx")])
    i=pl.scan_parquet(ITEM_MAP).select([pl.col(IID).alias("_i"),pl.col(IIX).cast(pl.UInt32).alias("item_idx")])
    w=((pl.col(SCORE).cast(pl.Float32,strict=False).fill_null(0).clip(0,1e9)+1).log().clip(0,10)).alias("weight")
    lf=(pl.scan_parquet(CF)
        .join(u,left_on=CFU,right_on="_u",how="inner")
        .join(i,left_on=CFI,right_on="_i",how="inner")
        .select(["user_idx","item_idx",w]))
    tmp=Path(str(TRAIN)+".tmp")
    if tmp.exists(): tmp.unlink()
    lf.sink_parquet(tmp,compression="zstd",row_group_size=250000,maintain_order=False)
    os.replace(tmp,TRAIN)
    atomic_json({"rows":pq.ParquetFile(TRAIN).metadata.num_rows},OK)
print("TRAIN:",pq.ParquetFile(TRAIN).metadata.num_rows); res()

TRAIN: 109718519
RAM available=26.31GB | working free=18.94GB


In [6]:
# Cell 6 — Build aligned hashed item metadata cache

fn = schema(CAT)
MID = col(fn, ["item_id", IID])

cand = {
    "product_id": ["product_id"],
    "brand": ["brand"],
    "category0": ["category0", "category_0"],
    "category1": ["category1", "category_1"],
    "category2": ["category2", "category_2"],
    "condition": ["condition"],
    "shipper": ["shipper"],
    "price": ["price"]
}

MC = {
    k: col(fn, v, False)
    for k, v in cand.items()
}

HASHED = ROOT / "processed" / "item_features_hashed.parquet"
HOK = ROOT / "status" / "_HASHED_SUCCESS.json"


# ============================================================
# 1. BUILD HASHED ITEM METADATA
# ============================================================

if not (valid_parquet(HASHED, 1000) and HOK.exists()):

    print("🚀 Building aligned hashed item metadata...")

    # Columns actually available
    use = [MID] + [
        x for x in MC.values()
        if x is not None
    ]

    # One metadata row / item
    meta = (
        pl.scan_parquet(CAT)
        .select(use)
        .unique(
            subset=[MID],
            keep="last"
        )
    )

    # TRAIN item universe
    im = (
        pl.scan_parquet(ITEM_MAP)
        .select([
            pl.col(IID)
            .alias("_id"),

            pl.col(IIX)
            .cast(pl.UInt32)
            .alias("item_idx")
        ])
    )

    # Keep all TRAIN items
    j = im.join(
        meta,
        left_on="_id",
        right_on=MID,
        how="left"
    )

    # --------------------------------------------------------
    # Hashed categorical features
    # --------------------------------------------------------

    ex = [
        pl.col("item_idx")
    ]

    for f, buckets in CFG["hash_buckets"].items():

        source_col = MC.get(f)

        if source_col is None:

            value_expr = pl.lit("__UNK__")

        else:

            value_expr = (
                pl.col(source_col)
                .cast(
                    pl.Utf8,
                    strict=False
                )
                .fill_null("__UNK__")
            )

        ex.append(
            (
                value_expr
                .hash(seed=SEED)
                % int(buckets)
            )
            .cast(pl.UInt32)
            .alias(f)
        )

    # --------------------------------------------------------
    # Price
    # --------------------------------------------------------

    pc = MC.get("price")

    if pc is None:

        p = pl.lit(
            0.0,
            dtype=pl.Float32
        )

    else:

        p = (
            (
                pl.col(pc)
                .cast(
                    pl.Float32,
                    strict=False
                )
                .fill_null(0.0)
                .clip(0.0, 1e8)
                + 1.0
            )
            .log()
            .cast(pl.Float32)
        )

    # --------------------------------------------------------
    # FIX:
    # mean và std phải có alias KHÁC NHAU
    # --------------------------------------------------------

    st = (
        j
        .select(
            p.alias("_price_log")
        )
        .select([
            pl.col("_price_log")
            .mean()
            .alias("price_mean"),

            pl.col("_price_log")
            .std()
            .alias("price_std")
        ])
        .collect(
            engine="streaming"
        )
    )

    pm = st["price_mean"][0]
    ps = st["price_std"][0]

    pm = float(
        pm
        if pm is not None
        else 0.0
    )

    ps = float(
        ps
        if ps is not None
        else 1.0
    )

    if not np.isfinite(pm):
        pm = 0.0

    if (
        not np.isfinite(ps)
        or ps < 1e-6
    ):
        ps = 1.0

    print(
        f"Price log mean={pm:.6f} "
        f"| std={ps:.6f}"
    )

    # --------------------------------------------------------
    # Final feature table
    # --------------------------------------------------------

    out = (
        j
        .select(
            ex
            + [
                p.alias("_p")
            ]
        )
        .with_columns(
            (
                (
                    pl.col("_p")
                    - pm
                )
                / ps
            )
            .cast(pl.Float32)
            .alias("price_z")
        )
        .drop("_p")
    )

    # --------------------------------------------------------
    # Safe write
    # --------------------------------------------------------

    tmp = Path(
        str(HASHED) + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    out.sink_parquet(
        tmp,
        compression="zstd",
        row_group_size=500_000,
        maintain_order=False
    )

    os.replace(
        tmp,
        HASHED
    )

    rows = (
        pq.ParquetFile(HASHED)
        .metadata
        .num_rows
    )

    atomic_json(
        {
            "rows": int(rows),
            "price_mean": pm,
            "price_std": ps
        },
        HOK
    )

    print(
        f"✅ HASHED metadata ready: "
        f"{rows:,} items"
    )

else:

    print(
        "✅ Reuse hashed metadata:",
        HASHED
    )


# ============================================================
# 2. MEMORY-MAPPED ITEM FEATURES
# ============================================================

FEATS = list(
    CFG["hash_buckets"]
)

MP = {
    f:
        ROOT
        / "features"
        / f"{f}.npy"

    for f in FEATS
}

MP["price_z"] = (
    ROOT
    / "features"
    / "price_z.npy"
)

MOK = (
    ROOT
    / "status"
    / "_MMAP_SUCCESS.json"
)


# ------------------------------------------------------------
# Validate mmap cache
# ------------------------------------------------------------

def mmap_cache_valid():

    if not MOK.exists():
        return False

    if not all(
        p.exists()
        for p in MP.values()
    ):
        return False

    try:

        info = json.loads(
            MOK.read_text()
        )

        return (
            int(
                info.get(
                    "n_items",
                    -1
                )
            )
            == N_ITEMS
        )

    except Exception:

        return False


# ============================================================
# 3. BUILD MMAP
# ============================================================

if not mmap_cache_valid():

    print(
        "🚀 Building item feature memmaps..."
    )

    mm = {}

    # categorical features
    for f in FEATS:

        mm[f] = (
            np.lib.format.open_memmap(
                MP[f],
                mode="w+",
                dtype=np.uint32,
                shape=(N_ITEMS,)
            )
        )

        mm[f][:] = 0

    # price
    mm["price_z"] = (
        np.lib.format.open_memmap(
            MP["price_z"],
            mode="w+",
            dtype=np.float32,
            shape=(N_ITEMS,)
        )
    )

    mm["price_z"][:] = 0.0


    pf = pq.ParquetFile(
        HASHED
    )

    done = 0

    read_columns = (
        ["item_idx"]
        + FEATS
        + ["price_z"]
    )


    for batch in pf.iter_batches(
        batch_size=500_000,
        columns=read_columns
    ):

        ix = (
            batch
            .column("item_idx")
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.int64,
                copy=False
            )
        )

        # categorical features
        for f in FEATS:

            mm[f][ix] = (
                batch
                .column(f)
                .to_numpy(
                    zero_copy_only=False
                )
                .astype(
                    np.uint32,
                    copy=False
                )
            )

        # price
        mm["price_z"][ix] = (
            batch
            .column("price_z")
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.float32,
                copy=False
            )
        )

        done += len(ix)

        if (
            done % 5_000_000
            < 500_000
        ):

            print(
                f"  {done:,}"
                f"/{N_ITEMS:,}"
            )


    # Flush to disk
    for a in mm.values():
        a.flush()

    del mm

    gc.collect()


    atomic_json(
        {
            "n_items": int(N_ITEMS),
            "features":
                FEATS
                + ["price_z"]
        },
        MOK
    )

    print(
        "✅ Item feature memmaps ready"
    )

else:

    print(
        "✅ Reuse existing item memmaps"
    )


# ============================================================
# 4. OPEN MEMORY MAP READ-ONLY
# ============================================================

ITEM_MM = {
    k:
        np.load(
            v,
            mmap_mode="r"
        )

    for k, v
    in MP.items()
}


print(
    {
        k: (
            v.shape,
            str(v.dtype)
        )

        for k, v
        in ITEM_MM.items()
    }
)


print(
    "✅ Item cache ready"
)

res()

🚀 Building aligned hashed item metadata...
Price log mean=1.528013 | std=1.760008
✅ HASHED metadata ready: 27,328,461 items
🚀 Building item feature memmaps...
  5,000,000/27,328,461
  10,000,000/27,328,461
  15,000,000/27,328,461
  20,000,000/27,328,461
  25,000,000/27,328,461
✅ Item feature memmaps ready
{'product_id': ((27328461,), 'uint32'), 'brand': ((27328461,), 'uint32'), 'category0': ((27328461,), 'uint32'), 'category1': ((27328461,), 'uint32'), 'category2': ((27328461,), 'uint32'), 'condition': ((27328461,), 'uint32'), 'shipper': ((27328461,), 'uint32'), 'price_z': ((27328461,), 'float32')}
✅ Item cache ready
RAM available=29.34GB | working free=17.90GB


In [7]:
# Cell 7 — Map VAL / TEST to TRAIN universe
def build_eval(split):
    files=sorted((INTDIR/f"split={split}").glob("*.parquet"))
    if not files: raise FileNotFoundError(split)
    out=ROOT/"processed"/f"{split}_mapped.parquet"
    ok=ROOT/"status"/f"_{split.upper()}_SUCCESS.json"
    if valid_parquet(out,100) and ok.exists(): return out
    n=schema(files[0]); u=col(n,["user_id","userid","user"]); i=col(n,["item_id","itemid","item"])
    e=col(n,["event_group","event","event_type","action"],False)
    su=pl.scan_parquet(USER_MAP).select([pl.col(UID).alias("_u"),pl.col(UIX).cast(pl.UInt32).alias("user_idx")])
    si=pl.scan_parquet(ITEM_MAP).select([pl.col(IID).alias("_i"),pl.col(IIX).cast(pl.UInt32).alias("item_idx")])
    x=pl.scan_parquet(str(INTDIR/f"split={split}" / "*.parquet")).join(su,left_on=u,right_on="_u",how="inner").join(si,left_on=i,right_on="_i",how="left")
    ev=pl.lit("") if not e else pl.col(e).cast(pl.Utf8,strict=False).fill_null("").str.to_lowercase()
    outlf=x.select([
        "user_idx","item_idx",
        ev.is_in(["like","item_like","cart","item_add_to_cart","offer","offer_make","buy_start","buy_comp","purchase"]).cast(pl.UInt8).alias("is_positive"),
        ev.is_in(["cart","item_add_to_cart","offer","offer_make","buy_start","buy_comp","purchase"]).cast(pl.UInt8).alias("is_strong"),
        ev.is_in(["buy_comp","purchase"]).cast(pl.UInt8).alias("is_purchase")
    ])
    tmp=Path(str(out)+".tmp")
    if tmp.exists(): tmp.unlink()
    outlf.sink_parquet(tmp,compression="zstd",row_group_size=250000,maintain_order=False)
    os.replace(tmp,out); atomic_json({"rows":pq.ParquetFile(out).metadata.num_rows},ok)
    return out

VAL=build_eval("val"); TEST=build_eval("test")
print("VAL/TEST ready"); res()

VAL/TEST ready
RAM available=29.33GB | working free=17.74GB


In [8]:
CFG["epochs"] = 15
print("Max epochs =", CFG["epochs"])

Max epochs = 15


In [9]:
# Cell 8 — Two-Tower model + optimizers
class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        D=CFG["embed_dim"]; Fd=CFG["feature_dim"]; H=CFG["hidden_dim"]
        self.user=nn.Embedding(N_USERS,D,sparse=True)
        self.meta=nn.ModuleDict({k:nn.Embedding(v,Fd,sparse=True) for k,v in CFG["hash_buckets"].items()})
        self.mlp=nn.Sequential(nn.Linear(len(self.meta)*Fd+1,H),nn.ReLU(),nn.Dropout(CFG["dropout"]),nn.Linear(H,D))
        nn.init.normal_(self.user.weight,std=.02)
        for e in self.meta.values(): nn.init.normal_(e.weight,std=.02)
    def encode_user(self,u): return F.normalize(self.user(u),dim=-1)
    def encode_item(self,x):
        z=[self.meta[k](x[k]) for k in self.meta]+[x["price_z"].unsqueeze(-1)]
        return F.normalize(self.mlp(torch.cat(z,dim=-1)),dim=-1)

model=TwoTower().to(DEVICE)
sparse=[model.user.weight]+[e.weight for e in model.meta.values()]
dense=list(model.mlp.parameters())
opt_s=SGD(sparse,lr=CFG["user_lr"])
opt_d=AdamW(dense,lr=CFG["dense_lr"],weight_decay=CFG["weight_decay"])

def item_feats(ids):
    ids=np.asarray(ids,dtype=np.int64); d={}
    for f in FEATS: d[f]=torch.from_numpy(np.asarray(ITEM_MM[f][ids],dtype=np.int64)).to(DEVICE)
    d["price_z"]=torch.from_numpy(np.asarray(ITEM_MM["price_z"][ids],dtype=np.float32)).to(DEVICE)
    return d

LATEST=ROOT/"checkpoints"/"twotower_latest.pt"; BEST=ROOT/"checkpoints"/"twotower_best.pt"
epoch0=0; rg0=0; step=0; best_metric=-1.0; bad=0; history=[]
if LATEST.exists():
    ck=torch.load(LATEST,map_location=DEVICE,weights_only=False)
    model.load_state_dict(ck["model"]); opt_s.load_state_dict(ck["opt_s"]); opt_d.load_state_dict(ck["opt_d"])
    epoch0=ck["epoch"]; rg0=ck["rg"]; step=ck["step"]; best_metric=ck["best"]; bad=ck["bad"]; history=ck["history"]
    print("RESUME:",epoch0,rg0,step)

def save_latest(ep,rg):
    atomic_torch({"model":model.state_dict(),"opt_s":opt_s.state_dict(),"opt_d":opt_d.state_dict(),
                  "epoch":ep,"rg":rg,"step":step,"best":best_metric,"bad":bad,"history":history},LATEST)
print(model); res()

TwoTower(
  (user): Embedding(2581378, 64, sparse=True)
  (meta): ModuleDict(
    (product_id): Embedding(1048576, 16, sparse=True)
    (brand): Embedding(262144, 16, sparse=True)
    (category0): Embedding(65536, 16, sparse=True)
    (category1): Embedding(65536, 16, sparse=True)
    (category2): Embedding(65536, 16, sparse=True)
    (condition): Embedding(1024, 16, sparse=True)
    (shipper): Embedding(65536, 16, sparse=True)
  )
  (mlp): Sequential(
    (0): Linear(in_features=113, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
  )
)
RAM available=28.64GB | working free=17.74GB


In [10]:
# Cell 9 — Evaluation helpers / fixed VAL sample
def sample_eval(path,n,seed):
    us=pl.scan_parquet(path).select("user_idx").unique().collect(engine="streaming")["user_idx"].to_numpy().astype(np.int64)
    rng=np.random.default_rng(seed); us=rng.choice(us,min(n,len(us)),replace=False)
    df=pl.scan_parquet(path).filter(pl.col("user_idx").is_in(us.tolist())).collect(engine="streaming")
    return us,df

def targets(df):
    T={m:{} for m in ["ALL","POSITIVE","STRONG","PURCHASE"]}; C={m:{} for m in T}
    for r in df.iter_rows(named=True):
        u=int(r["user_idx"]); it=r["item_idx"]
        fl={"ALL":1,"POSITIVE":r["is_positive"],"STRONG":r["is_strong"],"PURCHASE":r["is_purchase"]}
        for m,a in fl.items():
            if not a: continue
            C[m][u]=C[m].get(u,0)+1
            if it is not None: T[m].setdefault(u,set()).add(int(it))
    return T,C

def seen(users):
    df=pl.scan_parquet(TRAIN).filter(pl.col("user_idx").is_in(users.tolist())).select(["user_idx","item_idx"]).collect(engine="streaming")
    d={}
    for u,i in df.iter_rows(): d.setdefault(int(u),set()).add(int(i))
    return d

def encode_items(ids,b=65536):
    model.eval(); out=[]
    with torch.no_grad():
        for s in range(0,len(ids),b):
            x=np.asarray(ids[s:s+b],dtype=np.int64); out.append(model.encode_item(item_feats(x)).cpu().numpy().astype(np.float32))
    return np.concatenate(out)

def ndcg(pred,t,k):
    dc=sum((1/math.log2(r+1)) for r,x in enumerate(pred[:k],1) if x in t)
    idc=sum(1/math.log2(r+1) for r in range(1,min(len(t),k)+1))
    return dc/idc if idc else 0.

def evaluate(index,users,T,C,S):
    maxk=max(CFG["eval_ks"])
    with torch.no_grad(): q=model.encode_user(torch.from_numpy(users).to(DEVICE)).cpu().numpy().astype(np.float32)
    _,ids=index.search(q,maxk+CFG["retrieve_extra"])
    R={}
    for rr,u in enumerate(users):
        banned=S.get(int(u),set()); z=[]
        for x in ids[rr]:
            x=int(x)
            if x>=0 and x not in banned: z.append(x)
            if len(z)>=maxk: break
        R[int(u)]=z
    O={}
    for m,tmap in T.items():
        wa=sum(len(x) for x in tmap.values()); al=sum(C[m].values())
        row={"targets_all":al,"targets_warm":wa,"warm_target_coverage":wa/al if al else 0}
        for k in CFG["eval_ks"]:
            a=[];p=[];h=[];n=[]
            for u in users:
                t=tmap.get(int(u),set())
                if not t: continue
                pr=R[int(u)][:k]; hit=sum(x in t for x in pr)
                a.append(hit/len(t)); p.append(hit/k); h.append(float(hit>0)); n.append(ndcg(pr,t,k))
            row[f"Recall@{k}"]=float(np.mean(a)) if a else 0
            row[f"Precision@{k}"]=float(np.mean(p)) if p else 0
            row[f"HitRate@{k}"]=float(np.mean(h)) if h else 0
            row[f"NDCG@{k}"]=float(np.mean(n)) if n else 0
        O[m]=row
    return O

VU,VDF=sample_eval(VAL,CFG["val_users"],SEED+1); VT,VC=targets(VDF); VS=seen(VU)
rng=np.random.default_rng(SEED+2)
ri=rng.choice(N_ITEMS,min(CFG["val_random_items"],N_ITEMS),replace=False)
gt=np.asarray([x for d in VT.values() for s in d.values() for x in s],dtype=np.int64)
VCANDS=np.unique(np.concatenate([ri,gt])) if len(gt) else ri
print("VAL users",len(VU),"candidates",len(VCANDS))

VAL users 3000 candidates 530913


In [11]:
# Cell 10 — Train / checkpoint / VAL selection
pf=pq.ParquetFile(TRAIN); NR=pf.num_row_groups

def val_metric():
    v=encode_items(VCANDS)
    nlist=min(1024,max(64,len(VCANDS)//1000))
    q=faiss.IndexFlatIP(CFG["embed_dim"]); ix=faiss.IndexIVFFlat(q,CFG["embed_dim"],nlist,faiss.METRIC_INNER_PRODUCT)
    tr=np.random.default_rng(SEED+3).choice(len(v),min(len(v),max(50000,nlist*40)),replace=False)
    ix.train(v[tr]); ix.add_with_ids(v,VCANDS); ix.nprobe=min(24,nlist)
    m=evaluate(ix,VU,VT,VC,VS); del ix,v; gc.collect(); return m

for ep in range(epoch0,CFG["epochs"]):
    model.train(); order=np.random.default_rng(SEED+ep).permutation(NR)
    start=rg0 if ep==epoch0 else 0; loss_sum=0.; nb=0; last=time.time()
    print("="*80); print("EPOCH",ep+1,"/",CFG["epochs"],"resume_rg",start)
    for pos in range(start,NR):
        rg=int(order[pos]); tb=pf.read_row_group(rg,columns=["user_idx","item_idx","weight"])
        u=tb.column("user_idx").to_numpy(zero_copy_only=False).astype(np.int64)
        it=tb.column("item_idx").to_numpy(zero_copy_only=False).astype(np.int64)
        w=tb.column("weight").to_numpy(zero_copy_only=False).astype(np.float32)
        perm=np.random.default_rng(SEED+ep*1000003+rg).permutation(len(u))
        for s in range(0,len(perm),CFG["batch_size"]):
            j=perm[s:s+CFG["batch_size"]]; ub=u[j]; ib=it[j]; wb=w[j]
            # unique user/item for cleaner in-batch negatives
            ku=np.zeros(len(ub),bool); ki=np.zeros(len(ib),bool)
            ku[np.unique(ub,return_index=True)[1]]=1; ki[np.unique(ib,return_index=True)[1]]=1
            keep=ku&ki; ub=ub[keep]; ib=ib[keep]; wb=wb[keep]
            if len(ub)<2: continue
            ut=torch.from_numpy(ub).to(DEVICE); ft=item_feats(ib); wt=torch.from_numpy(wb).to(DEVICE)
            U=model.encode_user(ut); V=model.encode_item(ft)
            logits=(U@V.T)/CFG["temperature"]; lab=torch.arange(len(ub),device=DEVICE)
            l1=F.cross_entropy(logits,lab,reduction="none"); l2=F.cross_entropy(logits.T,lab,reduction="none")
            ww=(wt/(wt.mean()+1e-6)).clamp(.25,4)
            loss=.5*((l1*ww).mean()+(l2*ww).mean())
            opt_s.zero_grad(set_to_none=True); opt_d.zero_grad(set_to_none=True)
            loss.backward(); torch.nn.utils.clip_grad_norm_(dense,CFG["grad_clip"]); opt_s.step(); opt_d.step()
            loss_sum+=float(loss.detach().cpu()); nb+=1; step+=1
        if (pos+1)%10==0 or pos+1==NR:
            print(f"rg {pos+1}/{NR} step={step:,} loss={loss_sum/max(nb,1):.5f}")
        if time.time()-last>=CFG["checkpoint_minutes"]*60:
            save_latest(ep,pos+1); print("✅ checkpoint",pos+1,"/",NR); res(); last=time.time()
        del tb,u,it,w,perm; gc.collect()

    vm=val_metric(); score=float(vm["POSITIVE"]["NDCG@20"])
    history.append({"epoch":ep+1,"loss":loss_sum/max(nb,1),"val":vm,"selection":score})
    atomic_json(history,ROOT/"metrics"/"train_history.json")
    print(json.dumps(vm,indent=2))
    if score>best_metric:
        best_metric=score; bad=0
        atomic_torch({"model":model.state_dict(),"epoch":ep+1,"score":score,"cfg":CFG},BEST)
        print("🏆 NEW BEST",score)
    else:
        bad+=1
    rg0=0; save_latest(ep+1,0)
    if bad>=CFG["patience"]:
        print("Early stop"); break

print("✅ TRAINING COMPLETE")

EPOCH 1 / 15 resume_rg 0
rg 10/439 step=1,230 loss=8.02749
rg 20/439 step=2,460 loss=7.91973
rg 30/439 step=3,690 loss=7.85964
rg 40/439 step=4,920 loss=7.81933
rg 50/439 step=6,150 loss=7.78769
rg 60/439 step=7,380 loss=7.76148
rg 70/439 step=8,610 loss=7.73789
rg 80/439 step=9,824 loss=7.71476
rg 90/439 step=11,054 loss=7.69077
rg 100/439 step=12,284 loss=7.66666
rg 110/439 step=13,514 loss=7.64173
rg 120/439 step=14,744 loss=7.61651
rg 130/439 step=15,974 loss=7.58941
rg 140/439 step=17,204 loss=7.56162
rg 150/439 step=18,434 loss=7.53363
rg 160/439 step=19,664 loss=7.50569
rg 170/439 step=20,894 loss=7.47900
rg 180/439 step=22,124 loss=7.45133
rg 190/439 step=23,354 loss=7.42463
rg 200/439 step=24,584 loss=7.39687
rg 210/439 step=25,814 loss=7.37106
rg 220/439 step=27,044 loss=7.34414
rg 230/439 step=28,274 loss=7.31768
rg 240/439 step=29,504 loss=7.29091
rg 250/439 step=30,734 loss=7.26410
rg 260/439 step=31,964 loss=7.23907
rg 270/439 step=33,194 loss=7.21381
rg 280/439 step=34,4

In [12]:
# Cell 11 — Save final BEST model
best=torch.load(BEST,map_location=DEVICE,weights_only=False)
model.load_state_dict(best["model"]); model.eval()
FINAL=ROOT/"model"/"two_tower_best_model.pt"
atomic_torch({"model":model.state_dict(),"cfg":CFG,"n_users":N_USERS,"n_items":N_ITEMS,
              "best_epoch":best["epoch"],"selection_value":best["score"]},FINAL)
print("✅ FINAL MODEL saved:",FINAL); res()

✅ FINAL MODEL saved: /kaggle/working/merrec_models/TwoTower/model/two_tower_best_model.pt
RAM available=28.72GB | working free=15.61GB


In [13]:
# Cell 12 — Build FULL FAISS IVF-PQ over all TRAIN items
FP=ROOT/"index"/"items_ivfpq.faiss"; FOK=ROOT/"status"/"_FAISS_SUCCESS.json"
if not (FP.exists() and FOK.exists()):
    D=CFG["embed_dim"]; nlist=min(CFG["faiss_nlist"],max(256,N_ITEMS//1000))
    quant=faiss.IndexFlatIP(D)
    ix=faiss.IndexIVFPQ(quant,D,nlist,CFG["faiss_m"],CFG["faiss_nbits"],faiss.METRIC_INNER_PRODUCT)
    rng=np.random.default_rng(SEED+10)
    sid=rng.choice(N_ITEMS,min(CFG["faiss_train_sample"],N_ITEMS),replace=False).astype(np.int64)
    sv=encode_items(sid,CFG["faiss_batch"]); ix.train(sv); del sid,sv; gc.collect()
    B=CFG["faiss_batch"]
    for s in range(0,N_ITEMS,B):
        ids=np.arange(s,min(s+B,N_ITEMS),dtype=np.int64); v=encode_items(ids,B); ix.add_with_ids(v,ids)
        if (s//B)%20==0 or s+B>=N_ITEMS: print(f"FAISS {min(s+B,N_ITEMS):,}/{N_ITEMS:,}")
        del ids,v
    ix.nprobe=CFG["faiss_nprobe"]
    tmp=Path(str(FP)+".tmp"); faiss.write_index(ix,str(tmp)); os.replace(tmp,FP)
    atomic_json({"ntotal":int(ix.ntotal),"nlist":nlist,"nprobe":ix.nprobe},FOK)
    del ix; gc.collect()
print("✅ FAISS:",FP); res()

FAISS 65,536/27,328,461
FAISS 13,172,736/27,328,461
FAISS 14,483,456/27,328,461
FAISS 15,794,176/27,328,461
FAISS 17,104,896/27,328,461
FAISS 18,415,616/27,328,461
FAISS 19,726,336/27,328,461
FAISS 21,037,056/27,328,461
FAISS 22,347,776/27,328,461
FAISS 23,658,496/27,328,461
FAISS 24,969,216/27,328,461
FAISS 26,279,936/27,328,461
FAISS 27,328,461/27,328,461
✅ FAISS: /kaggle/working/merrec_models/TwoTower/index/items_ivfpq.faiss
RAM available=28.79GB | working free=15.00GB


In [14]:
# Cell 13 — Full VAL + FINAL TEST
index=faiss.read_index(str(FP)); index.nprobe=CFG["faiss_nprobe"]
VFULL=evaluate(index,VU,VT,VC,VS)
atomic_json(VFULL,ROOT/"metrics"/"val_full_index_metrics.json")

TU,TDF=sample_eval(TEST,CFG["test_users"],SEED+20); TT,TC=targets(TDF); TS=seen(TU)
TFULL=evaluate(index,TU,TT,TC,TS)
atomic_json(TFULL,ROOT/"metrics"/"test_full_index_metrics.json")

summary={
 "model":"TwoTower","best_epoch":best["epoch"],"selection_metric":"POSITIVE_NDCG@20",
 "selection_value":best["score"],"train_users":N_USERS,"train_items":N_ITEMS,
 "train_pairs":N_PAIRS,"val_users":len(VU),"test_users":len(TU),
 "faiss_ntotal":int(index.ntotal),"completed_at":datetime.now().isoformat()
}
atomic_json(summary,ROOT/"metrics"/"summary.json")
print("VAL FULL:",json.dumps(VFULL,indent=2))
print("TEST FULL:",json.dumps(TFULL,indent=2))

VAL FULL: {
  "ALL": {
    "targets_all": 51731,
    "targets_warm": 31646,
    "warm_target_coverage": 0.6117415089598114,
    "Recall@10": 0.0009268286234600599,
    "Precision@10": 0.0007084661707403471,
    "HitRate@10": 0.006730428622033298,
    "NDCG@10": 0.0009158909900096125,
    "Recall@20": 0.0017853155844227571,
    "Precision@20": 0.0007438894792773644,
    "HitRate@20": 0.01381509032943677,
    "NDCG@20": 0.0011922138404247323,
    "Recall@100": 0.007166691100083352,
    "Precision@100": 0.0005986539142755934,
    "HitRate@100": 0.04782146652497343,
    "NDCG@100": 0.0027229845788629887
  },
  "POSITIVE": {
    "targets_all": 7942,
    "targets_warm": 4924,
    "warm_target_coverage": 0.6199949634852682,
    "Recall@10": 0.0006132461161079313,
    "Precision@10": 0.0001635322976287817,
    "HitRate@10": 0.001635322976287817,
    "NDCG@10": 0.0005974352391838803,
    "Recall@20": 0.0014309076042518397,
    "Precision@20": 0.00012264922322158627,
    "HitRate@20": 0.00245298

In [15]:
# Cell 14 — Serving manifest + ZIP (giống ALS)
SC=ROOT/"serving"/"twotower_config.json"
README=ROOT/"serving"/"README_SERVING.txt"
atomic_json({
 "model_type":"TwoTower",
 "user_tower":"TRAIN user embedding",
 "item_tower":"hashed metadata embeddings + MLP",
 "similarity":"cosine/inner-product",
 "cold_user_fallback":"Popularity/Trending/Co-visitation/SASRec",
 "cfg":CFG
},SC)
README.write_text(
"""MerRec Two-Tower serving package

Flow:
user_id -> train_user_map -> user_idx -> user tower
-> FAISS -> item_idx -> train_item_map -> item_id -> catalog/database

TEST interactions are not used for training/model selection.
""",encoding="utf-8")

ZIP=WORK/"TwoTower_serving_package.zip"
tmp=Path(str(ZIP)+".tmp")
if tmp.exists(): tmp.unlink()

pack=[
 (FINAL,"model/two_tower_best_model.pt"),
 (FP,"index/items_ivfpq.faiss"),
 (USER_MAP,"mappings/train_user_map.parquet"),
 (ITEM_MAP,"mappings/train_item_map.parquet"),
 (SC,"twotower_config.json"),
 (README,"README_SERVING.txt"),
 (ROOT/"metrics"/"train_history.json","metrics/train_history.json"),
 (ROOT/"metrics"/"val_full_index_metrics.json","metrics/val_full_index_metrics.json"),
 (ROOT/"metrics"/"test_full_index_metrics.json","metrics/test_full_index_metrics.json"),
 (ROOT/"metrics"/"summary.json","metrics/summary.json"),
]
with zipfile.ZipFile(tmp,"w",allowZip64=True) as z:
    for src,arc in pack:
        src=Path(src)
        if src.exists():
            comp=zipfile.ZIP_DEFLATED if src.suffix.lower() in [".json",".txt"] else zipfile.ZIP_STORED
            z.write(src,arcname=arc,compress_type=comp)
os.replace(tmp,ZIP)

atomic_json({"state":"COMPLETE","serving_zip":str(ZIP),"completed_at":datetime.now().isoformat()},
            ROOT/"_COMPLETE.json")

print("✅ Serving ZIP:",ZIP)
print(f"ZIP size={ZIP.stat().st_size/2**30:.2f}GB")
print("🎉 TWO-TOWER PIPELINE COMPLETE")
res()

✅ Serving ZIP: /kaggle/working/TwoTower_serving_package.zip
ZIP size=1.42GB
🎉 TWO-TOWER PIPELINE COMPLETE
RAM available=28.14GB | working free=13.57GB


In [16]:
from pathlib import Path
import zipfile

zip_path = Path("/kaggle/working/TwoTower_serving_package.zip")

print("ZIP tồn tại:", zip_path.exists())

if zip_path.exists():
    print("ZIP size:", round(zip_path.stat().st_size / 1024**3, 2), "GB")

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

    print("\nCác file quan trọng:")
    for key in [
        "two_tower_best_model.pt",
        "items_ivfpq.faiss",
        "train_user_map.parquet",
        "train_item_map.parquet",
        "test_full_index_metrics.json",
    ]:
        matches = [x for x in names if x.endswith(key)]
        print(key, "=>", "✅" if matches else "❌", matches[:1])

ZIP tồn tại: True
ZIP size: 1.42 GB

Các file quan trọng:
two_tower_best_model.pt => ✅ ['model/two_tower_best_model.pt']
items_ivfpq.faiss => ✅ ['index/items_ivfpq.faiss']
train_user_map.parquet => ✅ ['mappings/train_user_map.parquet']
train_item_map.parquet => ✅ ['mappings/train_item_map.parquet']
test_full_index_metrics.json => ✅ ['metrics/test_full_index_metrics.json']
